# **08_Risk_Score.py**

Creates a 0-1 Crypto Scam Risk Score, grouped into Low, Medium, and High, based on the projected probability from the trained Random Forest models.

Thresholds:
    Low    < 0.3
    Medium 0.3 - < 0.7
    High   >= 0.7

Inputs:
    models/rf_elliptic.joblib
    models/rf_ethereum.joblib
    train_test_data/elliptic_test.csv
    train_test_data/ethereum_test.csv

Outputs:
    results/risk_scores_elliptic.csv
    results/risk_scores_ethereum.csv


In [1]:
import os
import pandas as pd
import numpy as np
import joblib


In [2]:
BASE_PATH = ".."

MODEL_PATH = os.path.join(BASE_PATH, "models")
DATA_PATH = os.path.join(BASE_PATH, "train_test_data")
RESULTS_PATH = os.path.join(BASE_PATH, "results")


In [3]:
LOW_MAX = 0.3
HIGH_MIN = 0.7

In [4]:
def risk_band(score):
    if score < LOW_MAX:
        return "Low"
    elif score < HIGH_MIN:
        return "Medium"
    return "High"

In [5]:
# ELLIPTIC

rf_ell = joblib.load("../models/rf_elliptic.joblib")
ell_test = pd.read_csv("../train_test_data/elliptic_test.csv")

feat_cols_ell = [c for c in ell_test.columns if c.startswith("feat_")]

In [6]:
scores_ell = rf_ell.predict_proba(ell_test[feat_cols_ell])[:, 1]
out_ell = ell_test[["txId", "label"]].copy()
out_ell["risk_score"] = scores_ell
out_ell["risk_band"] = [risk_band(s) for s in scores_ell]

In [7]:
print("Elliptic risk band distribution:")
print(out_ell["risk_band"].value_counts())
print("\nActual illicit rate within each band:")
print(out_ell.groupby("risk_band")["label"].mean())


Elliptic risk band distribution:
risk_band
Low       14896
Medium     1069
High        705
Name: count, dtype: int64

Actual illicit rate within each band:
risk_band
High      0.957447
Low       0.018394
Medium    0.125351
Name: label, dtype: float64


In [8]:
# ETHEREUM
rf_eth = joblib.load("../models/rf_ethereum.joblib")
eth_test = pd.read_csv("../train_test_data/ethereum_test.csv")
feat_cols_eth = [c for c in eth_test.columns if c not in ["Address", "FLAG"]]

In [9]:
scores_eth = rf_eth.predict_proba(eth_test[feat_cols_eth])[:, 1]
out_eth = eth_test[["Address", "FLAG"]].copy()
out_eth["risk_score"] = scores_eth
out_eth["risk_band"] = [risk_band(s) for s in scores_eth]

In [10]:
import os
import matplotlib.pyplot as plt

# Project paths
BASE_PATH = ".."
EDA_PATH = os.path.join(BASE_PATH, "eda_charts")

# Make sure the folder exists
os.makedirs(EDA_PATH, exist_ok=True)

# Risk band order
BAND_ORDER = ["Low", "Medium", "High"]

# Create figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

datasets = [
    (out_ell, "label", "Elliptic: Risk Score Validation", axes[0]),
    (out_eth, "FLAG", "Ethereum: Risk Score Validation", axes[1])
]

for df, label_col, title, ax in datasets:

    # Number of cases in each risk band
    counts = (
        df["risk_band"]
        .value_counts()
        .reindex(BAND_ORDER)
        .fillna(0)
    )

    # Actual fraud/illicit rate
    rates = (
        df.groupby("risk_band")[label_col]
        .mean()
        .reindex(BAND_ORDER)
    )

    # Bar chart
    ax.bar(
        BAND_ORDER,
        counts.values,
        width=0.6
    )

    # Counts above bars
    for i, count in enumerate(counts.values):
        ax.text(
            i,
            count + counts.max() * 0.02,
            f"n={int(count):,}",
            ha="center",
            fontsize=10
        )

    # Fraud rate line
    ax2 = ax.twinx()

    ax2.plot(
        BAND_ORDER,
        rates.values,
        marker="o",
        linewidth=2.5,
        markersize=8
    )

    # Percentage labels
    for i, rate in enumerate(rates.values):
        ax2.text(
            i,
            rate + 0.04,
            f"{rate:.1%}",
            ha="center",
            fontsize=10
        )

    ax.set_xlabel("Risk band")
    ax.set_ylabel("Number of transactions/accounts")
    ax2.set_ylabel("Actual fraud/illicit rate")
    ax2.set_ylim(0, 1.1)

    ax.set_title(title)

fig.tight_layout()

# Save chart
chart_path = os.path.join(
    EDA_PATH,
    "N_risk_band_validation.png"
)

fig.savefig(
    chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.close(fig)

print("Risk score chart saved to:")
print(os.path.abspath(chart_path))
print("File exists:", os.path.exists(chart_path))

Risk score chart saved to:
/Users/dhilnasherin/Downloads/Dissertation_dilna/ML_Pipeline/eda_charts/N_risk_band_validation.png
File exists: True


In [11]:
print("\nEthereum risk band distribution:")
print(out_eth["risk_band"].value_counts())
print("\nActual fraud rate within each band:")
print(out_eth.groupby("risk_band")["FLAG"].mean())


Ethereum risk band distribution:
risk_band
Low       1517
High       386
Medium      61
Name: count, dtype: int64

Actual fraud rate within each band:
risk_band
High      0.997409
Low       0.011866
Medium    0.540984
Name: FLAG, dtype: float64


In [12]:
# Save risk-score outputs 
os.makedirs(RESULTS_PATH, exist_ok=True)

out_ell.to_csv(os.path.join(RESULTS_PATH, "risk_scores_elliptic.csv"), index=False)
out_eth.to_csv(os.path.join(RESULTS_PATH, "risk_scores_ethereum.csv"), index=False)

print("Risk-score files saved successfully!")
print("\nSaved files:")
print(os.path.join(RESULTS_PATH, "risk_scores_elliptic.csv"))
print(os.path.join(RESULTS_PATH, "risk_scores_ethereum.csv"))

Risk-score files saved successfully!

Saved files:
../results/risk_scores_elliptic.csv
../results/risk_scores_ethereum.csv


In [13]:
results_path = "../results"

print(os.listdir(results_path))

['risk_scores_ethereum.csv', 'risk_scores_elliptic.csv']
